# beat2body: run the 2D pipeline on Colab (README 4.3-4.7)

Runs the three pose tools (`pose_mediapipe.py`, `pose_dwpose.py`, `pose_openpose.py` in `2d/scripts/`)
on every video of the chosen categories, then runs the later steps of `pipeline.py` for each tool.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

Steps:
1. Install packages.
2. Choose categories and download their videos from the team repository.
3. Run the three pose tools.
4. Clean the keypoints, extract audio features, cut clips and split (README 4.4-4.7) for each tool.

Results:
- `2d/keypoints/<tool>/<category>/<video_id>.npz`: cleaned keypoints of each tool
- `2d/clips/<tool>/train|val|test/` and `2d/clips/<tool>/meta.csv`: clips of each tool
- `2d/audio/`, `2d/features/`: audio and audio features, shared by the three tools

Use `Runtime -> Run all`.

In [ ]:
# 1. Check the GPU and install packages
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DWPose and OpenPose will run on CPU (slow)"
!pip -q install mediapipe onnxruntime-gpu
!pip -q install --no-deps easy-dwpose
# If pip reports that numpy was changed: Runtime -> Restart session, then continue from the next cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# 2. Choose categories and download their videos
CATEGORIES = ["la_hiphop"]
# All 10: la_hiphop, house, middle_hiphop, break, pop, lock, waack, krump, street_jazz, ballet_jazz
LIMIT = None      # videos per category for a quick test; None = all 21
REPO = "/content/beat2body"
SOURCE = "https://github.com/SAMMYBOOOOM/beat2body"
BRANCH = "main"

import os, glob, subprocess
if not os.path.exists(REPO):
    !git clone -q --depth 1 --filter=blob:none --sparse -b {BRANCH} {SOURCE} {REPO}
else:
    !cd {REPO} && git pull -q origin {BRANCH}
names = subprocess.run(["git", "ls-tree", "-r", "--name-only", "HEAD", "2d/raw_video"],
                       cwd=REPO, capture_output=True, text=True).stdout.splitlines()
VIDEOS = []
for c in CATEGORIES:
    vids = sorted(n[len("2d/raw_video/"):] for n in names if n.startswith(f"2d/raw_video/{c}/") and n.endswith(".mp4"))
    VIDEOS += vids[:LIMIT] if LIMIT else vids
patterns = " ".join(["'/*'", "'!/2d/raw_video/**/*.mp4'"] + [f"'/2d/raw_video/{v}'" for v in VIDEOS])
!cd {REPO} && git sparse-checkout set --no-cone {patterns}
for v in VIDEOS:
    print("OK     " if os.path.exists(f"{REPO}/2d/raw_video/{v}") else "MISSING", v)
print("scripts:", sorted(os.listdir(f"{REPO}/2d/scripts")))

In [ ]:
# 3a. MediaPipe (CPU)
# MediaPipe's GPU mode needs OpenGL, which Colab does not provide, so it runs on CPU here.
# Videos that already have output are skipped; add --overwrite to run them again.
%cd {REPO}
!python 2d/scripts/pose_mediapipe.py --device cpu

In [ ]:
# 3b. DWPose (GPU)
%cd {REPO}
!python 2d/scripts/pose_dwpose.py

In [ ]:
# 3c. OpenPose (GPU, official CMU build)
# The first run compiles OpenPose (about 25 minutes) and saves the build to your Google Drive;
# later sessions restore it from there in a few minutes.
CACHE = "/content/drive/MyDrive/openpose_colab_build.tar.gz"
BIN = "/content/openpose/build/examples/openpose/openpose.bin"
if not os.path.exists(BIN) and os.path.exists(CACHE):
    print("restoring OpenPose from Google Drive")
    !tar xzf {CACHE} -C /content
    !apt-get -qq update > /dev/null && apt-get -qq install -y libgoogle-glog-dev libgflags-dev libprotobuf-dev libhdf5-serial-dev libatlas-base-dev libboost-all-dev libopencv-dev > /dev/null
!bash {REPO}/2d/scripts/colab_build_openpose.sh
if os.path.exists(BIN) and not os.path.exists(CACHE):
    print("saving the OpenPose build to Google Drive")
    !tar czf {CACHE} -C /content --exclude='*.o' --exclude='.git' openpose
%cd {REPO}
!python 2d/scripts/pose_openpose.py --backend official

In [ ]:
# 4. README 4.4-4.7 for each of the three tools, keeping only the cleaned keypoints:
#    keypoints/<tool>/<category>/<video_id>.npz   cleaned keypoints (4.4)
#    keypoints/<tool>/normalization.json
#    clips/<tool>/train|val|test/, clips/<tool>/meta.csv   (4.6 + 4.7)
#    audio/ and features/ (4.5) are shared by the three tools.
# The raw pose outputs (*_<tool>.npz) are deleted at the end.
import glob, os, shutil, subprocess, sys

def run(*args):
    r = subprocess.run([sys.executable, "2d/scripts/pipeline.py", *args], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()

TOOLS = ["dwpose", "mediapipe", "openpose"]
K, C = f"{REPO}/2d/keypoints", f"{REPO}/2d/clips"
CATS = sorted({v.split("/")[0] for v in VIDEOS})

run("audio", "--categories", *CATS)
for tool in TOOLS:
    raw = [f for c in CATS for f in glob.glob(f"{K}/{c}/*_{tool}.npz")]
    if not raw:
        print(f"[{tool}] no pose output, skipped")
        continue
    run("clean", "--tool", tool, "--categories", *CATS)
    run("clips", "--categories", *CATS)
    for c in CATS:
        os.makedirs(f"{K}/{tool}/{c}", exist_ok=True)
        for f in glob.glob(f"{K}/{c}/*.npz"):
            name = os.path.basename(f)
            if not any(name.endswith(f"_{t}.npz") for t in TOOLS):
                shutil.move(f, f"{K}/{tool}/{c}/{name}")
    shutil.move(f"{K}/normalization.json", f"{K}/{tool}/normalization.json")
    for split in ["train", "val", "test"]:
        dst = f"{C}/{tool}/{split}"
        shutil.rmtree(dst, ignore_errors=True)
        os.makedirs(dst)
        for f in glob.glob(f"{C}/{split}/*.npz"):
            shutil.move(f, dst)
    meta = open(f"{REPO}/2d/meta.csv").read().replace("2d/clips/", f"2d/clips/{tool}/")
    open(f"{C}/{tool}/meta.csv", "w").write(meta)
    os.remove(f"{REPO}/2d/meta.csv")
    for f in raw:
        os.remove(f)
    print(f"[{tool}] done -> 2d/keypoints/{tool}/ and 2d/clips/{tool}/")

In [ ]:
# 5. Save the results to Google Drive
!mkdir -p /content/drive/MyDrive/beat2body_results && cp -r {REPO}/2d/keypoints {REPO}/2d/clips {REPO}/2d/features {REPO}/2d/categories.json /content/drive/MyDrive/beat2body_results/

In [ ]:
# 5b. Upload the results to the Hugging Face dataset (disabled until the dataset repo exists)
# To use it: add HF_TOKEN in Colab Secrets (key icon), set repo_id.
# from google.colab import userdata
# from huggingface_hub import HfApi
# api = HfApi(token=userdata.get("HF_TOKEN"))
# api.upload_folder(repo_id="<team>/beat2body-2d", repo_type="dataset",
#                   folder_path=f"{REPO}/2d", path_in_repo="2d",
#                   allow_patterns=["keypoints/**", "clips/**", "features/**", "categories.json"],
#                   ignore_patterns=["**/.gitkeep"])